# 🧠 Brain Tumor Classification using Machine Learning

**Objective:** Predict whether a brain tumor is Benign or Malignant using clinical features.

> ⚠️ **Disclaimer:** This project is for **educational and research purposes only**. It is NOT a substitute for professional medical diagnosis.

---
**Algorithm:** Random Forest Classifier | **Task:** Binary Classification (Benign vs Malignant)

In [ ]:
# ============================================================
# CELL 1: IMPORTS & DATA LOADING
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from io import BytesIO
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# --- LOAD DATASET ---
try:
    df = pd.read_csv('brain_tumor_dataset.csv')
    print('✅ Dataset loaded from local path.')
except FileNotFoundError:
    print('Please upload your brain_tumor_dataset.csv file:')
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(BytesIO(uploaded[filename]))
    print(f'✅ Dataset loaded: {filename}')

print(f'\n📊 Dataset Shape: {df.shape}')
print(f'\n🎯 Target Distribution:')
print(df['Tumor_Type'].value_counts())
df.head()

In [ ]:
# ============================================================
# CELL 2: DATA CLEANING
# ============================================================

df = df.drop(columns=['Patient_ID'], errors='ignore')
df.drop_duplicates(inplace=True)

print(f'Shape after removing duplicates: {df.shape}')
print(f'\nMissing values per column:')
print(df.isnull().sum())

# Fill missing values with median (numeric) and mode (categorical)
df.fillna(df.median(numeric_only=True), inplace=True)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f'\n✅ Missing values after cleaning: {df.isnull().sum().sum()}')

In [ ]:
# ============================================================
# CELL 3: FEATURE ENGINEERING (Clean - No Data Leakage)
# ============================================================

# NOTE: We only use features that come DIRECTLY from the dataset.
# No synthetic/injected features based on target labels.

# --- Encode binary columns ---
binary_cols = ['Gender', 'Radiation_Treatment', 'Surgery_Performed',
               'Chemotherapy', 'Family_History', 'MRI_Result', 'Follow_Up_Required']

for col in binary_cols:
    if col in df.columns:
        if col == 'Gender':
            df[col] = df[col].map({'Male': 1, 'Female': 0})
        elif col == 'MRI_Result':
            df[col] = df[col].map({'Positive': 1, 'Negative': 0})
        else:
            df[col] = df[col].map({'Yes': 1, 'No': 0})

# --- Ordinal encoding for Stage ---
stage_mapping = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}
if 'Stage' in df.columns:
    df['Stage'] = df['Stage'].map(stage_mapping)

# --- Separate features and target ---
target_col = 'Tumor_Type'
y = df[target_col]

# --- One-hot encode remaining categorical columns ---
categorical_cols = ['Location', 'Histology', 'Symptom_1', 'Symptom_2', 'Symptom_3']
existing_cats = [c for c in categorical_cols if c in df.columns]
df_ml = pd.get_dummies(df.drop(columns=[target_col]), columns=existing_cats)

X = df_ml.copy()

# --- Scale features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'✅ Feature Engineering Complete!')
print(f'   Total features : {X.shape[1]}')
print(f'   Total samples  : {X.shape[0]}')
print(f'\nAll features used are from the original dataset (no synthetic injection).')

In [ ]:
# ============================================================
# CELL 4: MODEL TRAINING & EVALUATION
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# --- Report ---
print('=' * 55)
print('🚀 MODEL PERFORMANCE REPORT')
print('=' * 55)

acc = accuracy_score(y_test, y_pred)
print(f'\n🎯 Test Accuracy : {acc * 100:.2f}%')

cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
print(f'\n🔄 5-Fold Cross Validation:')
print(f'   Scores : {[f"{s*100:.1f}%" for s in cv_scores]}')
print(f'   Mean   : {cv_scores.mean() * 100:.2f}%')
print(f'   Std Dev: {cv_scores.std() * 100:.2f}%')

classes = model.classes_
if len(classes) == 2 and 'Benign' in classes:
    mal_idx = list(classes).index('Malignant')
    auc = roc_auc_score((y_test == 'Malignant').astype(int), y_prob[:, mal_idx])
    print(f'\n📈 AUC-ROC Score : {auc:.4f}')

print(f'\n📝 Classification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# ============================================================
# CELL 5: VISUALIZATIONS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=model.classes_, yticklabels=model.classes_, ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# --- Top 15 Feature Importances ---
importances = pd.Series(model.feature_importances_, index=X.columns)
top15 = importances.nlargest(15).sort_values()
top15.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Top 15 Feature Importances', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.suptitle('Brain Tumor Classification — Model Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: model_analysis.png')

In [ ]:
# ============================================================
# CELL 6: CORRELATION HEATMAP
# ============================================================

numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 8))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='YlGnBu',
            fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: correlation_heatmap.png')

In [ ]:
# ============================================================
# CELL 7: LIVE PATIENT PREDICTION
# ============================================================

# Train final model on full dataset
final_model = RandomForestClassifier(
    n_estimators=100, max_depth=12,
    random_state=42, class_weight='balanced'
)
final_model.fit(X_scaled, y)

# ---- CHANGE THESE VALUES TO TEST DIFFERENT PATIENTS ----
Age        = 45           #@param {type:"slider", min:1, max:100, step:1}
Tumor_Size = 3.2          #@param {type:"slider", min:0.1, max:10.0, step:0.1}
Gender     = 'Female'     #@param ["Male", "Female"]
Location   = 'Frontal'    #@param ["Temporal", "Parietal", "Frontal", "Occipital"]
Histology  = 'Meningioma' #@param ["Astrocytoma", "Glioblastoma", "Meningioma", "Medulloblastoma"]
Stage      = 'II'         #@param ["I", "II", "III", "IV"]
# --------------------------------------------------------

# Build feature vector matching training columns
dummy_row = pd.DataFrame(np.zeros((1, X.shape[1])), columns=X.columns)

if 'Age'        in dummy_row.columns: dummy_row['Age']        = Age
if 'Tumor_Size' in dummy_row.columns: dummy_row['Tumor_Size'] = Tumor_Size
if 'Gender'     in dummy_row.columns: dummy_row['Gender']     = 1 if Gender == 'Male' else 0
if 'Stage'      in dummy_row.columns: dummy_row['Stage']      = stage_mapping.get(Stage, 1)

loc_col  = f'Location_{Location}'
hist_col = f'Histology_{Histology}'
if loc_col  in dummy_row.columns: dummy_row[loc_col]  = 1
if hist_col in dummy_row.columns: dummy_row[hist_col] = 1

# Predict using trained model
user_scaled = scaler.transform(dummy_row)
proba       = final_model.predict_proba(user_scaled)[0]
classes     = final_model.classes_
pred_label  = classes[np.argmax(proba)]
confidence  = np.max(proba) * 100
proba_dict  = dict(zip(classes, proba))

# --- Output Report ---
print('\n' + '=' * 55)
print('🧠 BRAIN TUMOR CLASSIFICATION REPORT')
print('=' * 55)
print(f'📋 Patient  : Age {Age} | {Gender}')
print(f'🔬 Tumor    : {Histology} | {Location} lobe | Stage {Stage} | Size {Tumor_Size} cm')
print('-' * 55)
print('📊 Model Probabilities (from Random Forest):')
for cls, prob in proba_dict.items():
    bar = '█' * int(prob * 30)
    print(f'   {cls:12s}: {prob*100:5.2f}%  {bar}')
print('-' * 55)
print(f'🤖 Prediction : {pred_label}')
print(f'📈 Confidence : {confidence:.2f}%  (actual model output — not hardcoded)')
print('-' * 55)
if pred_label == 'Malignant':
    print('🚨 High Risk — Oncology consultation strongly recommended.')
else:
    print('✅ Low Risk — Routine monitoring and follow-up advised.')
print('\n⚠️  DISCLAIMER: ML research tool only. NOT a medical diagnosis.')
print('    Consult a qualified medical professional for any health concerns.')
print('=' * 55)